# MN-GLM ~2B — GLM-5.2-class on the memory-native method (self-contained Colab)

A **~2B-parameter** GLM-5.2-class model (RMSNorm + GQA + RoPE + QK-norm + **SwiGLU Counter-MoE**) on the counter-synapse method: 0.75 B/weight, optimizer-in-state, reversible O(1) activations (`anchor_every=2`), int8 fwd+update. The `memory_native` package is **embedded** (base64) — no repo clone / pip install. Just: Runtime → GPU, then Run all.

The printed coeff count includes the MoE experts (they live in stacked buffers, ~93% of the weights). OOM on a small GPU? drop `BATCH` (memory scales with batch x seq).

In [ ]:
!pip -q install tiktoken datasets 2>/dev/null
print('deps ok')

In [ ]:
# self-contained: the memory_native package is embedded below as a base64 gzip tar,
# unpacked into sys.path -- no repo clone / pip install of the package needed.
import base64, io, tarfile, sys, os
PKG_B64 = "..."
os.makedirs("/content/mn", exist_ok=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(PKG_B64)), mode="r:gz").extractall("/content/mn")
sys.path.insert(0, "/content/mn")
import memory_native; print("memory_native ready:", memory_native.__file__)

In [ ]:
import glob, os, time, torch
from memory_native import ReversibleMNGLM, fmt_bytes, peak_training_memory
from memory_native.optimizers import build_optimizer

# ---- ~2B MN-GLM (D=1536, L=24, GQA 12/2, E=8/top2 SwiGLU). Counter weights are sub-byte + reversible
#      O(1) activations, so training peak is modest. If you OOM on a small GPU, drop BATCH.
D, L, NH, NKV, BLK = 1536, 24, 12, 2, 1024
E, TOP_K = 8, 2
LR, LR_SCALE, C, ABITS, ANCHOR, INT8 = 2e-3, 2e-4, 11, 8, 2, True
BATCH, LOSS_CHUNK, VAL_TOKENS = 8, 2048, 300_000
CKPT_EVERY, STEP_CAP, TIME_CAP, CKPT_OUT = 200, 200_000, 36_000, "/content/ckpt_glm.pt"

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU"
dev = "cuda"
import tiktoken
enc = tiktoken.get_encoding("gpt2"); vocab = enc.n_vocab

from datasets import load_dataset
class FWStream:
    def __init__(s):
        s.it = iter(load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True))
        s.eot = enc.eot_token; s.buf = []
    def _fill(s, n):
        while len(s.buf) < n:
            s.buf.extend(enc.encode_ordinary(next(s.it)["text"])); s.buf.append(s.eot)
    def pull(s, n, d):
        s._fill(n); o = torch.tensor(s.buf[:n], dtype=torch.long, device=d); s.buf = s.buf[n:]; return o
    def batch(s, B, T, d):
        need = B*(T+1); s._fill(need)
        c = torch.tensor(s.buf[:need], dtype=torch.long, device=d).view(B, T+1); s.buf = s.buf[need:]
        return c[:, :T].contiguous(), c[:, 1:].contiguous()

stream = FWStream(); val_ids = stream.pull(VAL_TOKENS, dev)
print(f"tokenizer gpt2 vocab {vocab}; val {val_ids.numel()/1e6:.2f}M tok", flush=True)
torch.manual_seed(0)
ckw = dict(lr=LR, lr_scale=LR_SCALE, C=C, act_save_bits=ABITS)
if INT8: ckw.update(forward_compute="int8", update_compute="int8")
m = ReversibleMNGLM(vocab, D, L, NH, NKV, BLK, kind="counter_packed", n_experts=E, top_k=TOP_K,
                    qk_norm=True, grouped=True, swiglu=True, anchor_every=ANCHOR, **ckw).to(dev).train()
opt = build_optimizer("adamw", m.trainable_parameters(), LR)
if os.path.exists(CKPT_OUT):
    st = torch.load(CKPT_OUT, map_location=dev); m.load_state_dict(st["model"]); opt.load_state_dict(st["opt"])
    start = st["step"]; print("RESUMED at", start, flush=True)
else:
    start = 0; print("fresh start", flush=True)
cc = m.total_counter_coeffs()   # attention counter linears + MoE experts (the bulk lives in stacked buffers)
fpp = sum(p.numel() for p in m.trainable_parameters())
print(f"MN-GLM d={D} L={L} GQA {NH}/{NKV} E={E}/{TOP_K} swiglu anchor={ANCHOR} int8={INT8}: "
      f"{cc/1e9:.2f}B counter coeffs (state ~{fmt_bytes(cc*3//4)}, 0.75 B/wt) + {fpp/1e6:.0f}M fp params "
      f"(embed/norms/router). vs dense+Adam on the counter weights: {fmt_bytes(cc*16)}", flush=True)
xb, yb = stream.batch(BATCH, BLK, dev)
def onestep():
    opt.zero_grad(set_to_none=True); _, l = m(xb, yb, loss_chunk=LOSS_CHUNK); l.backward(); opt.step()
print("one-step peak:", fmt_bytes(peak_training_memory(onestep, torch.device(dev))), flush=True)

@torch.no_grad()
def evaluate(n=20):
    m.eval(); tot = 0.0; g = torch.Generator(device=dev).manual_seed(0)
    for _ in range(n):
        i = torch.randint(0, val_ids.numel()-BLK-1, (BATCH,), generator=g, device=dev)
        x = torch.stack([val_ids[j:j+BLK] for j in i]); y = torch.stack([val_ids[j+1:j+1+BLK] for j in i])
        tot += m(x, y, loss_chunk=LOSS_CHUNK)[1].item()
    m.train(); return tot/n

print(f"\n==== TRAIN ~2B MN-GLM  batch {BATCH} x seq {BLK}  from step {start} ====", flush=True)
torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t0 = time.time(); s = start; toks = BATCH*BLK
while s < STEP_CAP and time.time()-t0 < TIME_CAP:
    x, y = stream.batch(BATCH, BLK, dev); _, loss = m(x, y, loss_chunk=LOSS_CHUNK)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); s += 1
    vl = evaluate() if s % 200 == 0 else None
    if s % 25 == 0:
        el = time.time()-t0; msg = f"  step {s:6d}  train {loss.item():.4f}  {(s-start)*toks/el:,.0f} tok/s  {el:.0f}s"
        if vl is not None: msg += f"  val {vl:.4f}"
        print(msg, flush=True)
    if s % CKPT_EVERY == 0:
        torch.save({"model": m.state_dict(), "opt": opt.state_dict(), "step": s}, CKPT_OUT)
print("peak", fmt_bytes(torch.cuda.max_memory_allocated()), flush=True)